# **Cross-Dataset Experiment (DAWN → ACDC) — Faster R-CNN/R50-FPN**
- Training Faster R-CNN on the DAWN dataset using COCO pretrained weights
- Evaluating the trained model on the global ACDC test set
- Performing weather-specific evaluations on the fog, rain, and snow subsets of the ACDC dataset

# Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Install Detectron

In [2]:
# Fix numpy compatibility
!pip install -q --force-reinstall numpy==1.26.4

# Build dependencies
!pip install -q setuptools==68.0.0 wheel cython

#Import detectron from the source
%cd /content

import os

if not os.path.exists("/content/detectron2"):
    !git clone https://github.com/facebookresearch/detectron2.git

%cd /content/detectron2
!python -m pip install --no-build-isolation -e .
%cd /content

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.52.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have n

In [ ]:
import os
os.kill(os.getpid(), 9)

# Imports

In [3]:
from pathlib import Path
import json
import os
import time

import cv2
import numpy as np
import pandas as pd
import torch

from detectron2.config import get_cfg
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import register_pascal_voc
from detectron2.data import build_detection_test_loader
from detectron2.engine import DefaultPredictor
from detectron2.evaluation import COCOEvaluator, inference_on_dataset

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print(
    "GPU:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
)

NumPy: 1.26.4
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB


# Path & Experiment settings

In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Dissertation")

# Existing prepared ACDC Pascal VOC dataset
ACDC_VOC_ROOT = PROJECT_ROOT / "Datasets/processed/acdc_voc"

# Existing DAWN seed-42 in-domain training directory
# Change only this line if your folder has a slightly different name.
SOURCE_RUN_DIR = (
    PROJECT_ROOT
    / "Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed42"
)

SOURCE_CONFIG = SOURCE_RUN_DIR / "config.yaml"
BEST_MODEL = SOURCE_RUN_DIR / "model_best.pth"
FINAL_MODEL = SOURCE_RUN_DIR / "model_final.pth"

# Keep cross-dataset outputs separate from the original training folder
OUTPUT_DIR = (
    PROJECT_ROOT
    / "Runs/faster_rcnn/dawn_to_acdc_faster_rcnn_seed42"
)

CLASS_NAMES = [
    "person",
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck",
]

SOURCE_DATASET = "DAWN"
TARGET_DATASET = "ACDC"
RANDOM_SEED = 42
SCORE_THRESHOLD = 0.5

print("ACDC VOC root exists:", ACDC_VOC_ROOT.exists())
print("Source run directory exists:", SOURCE_RUN_DIR.exists())
print("Saved config exists:", SOURCE_CONFIG.exists())
print("Best model exists:", BEST_MODEL.exists())
print("Final model exists:", FINAL_MODEL.exists())
print("Output directory:", OUTPUT_DIR)

assert ACDC_VOC_ROOT.exists(), f"Missing dataset: {ACDC_VOC_ROOT}"
assert SOURCE_RUN_DIR.exists(), f"Missing source run directory: {SOURCE_RUN_DIR}"
assert SOURCE_CONFIG.exists(), f"Missing saved config: {SOURCE_CONFIG}"
assert BEST_MODEL.exists() or FINAL_MODEL.exists(), (
    "Neither model_best.pth nor model_final.pth was found."
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ACDC VOC root exists: True
Source run directory exists: True
Saved config exists: True
Best model exists: True
Final model exists: True
Output directory: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_to_acdc_faster_rcnn_seed42


# Verify ACDC test splits

In [6]:
required_splits = ["test", "test_fog", "test_rain", "test_snow"]

for split_name in required_splits:
    split_file = (
        ACDC_VOC_ROOT
        / "ImageSets"
        / "Main"
        / f"{split_name}.txt"
    )
    print(f"{split_name}: {split_file.exists()} — {split_file}")
    assert split_file.exists(), f"Missing split file: {split_file}"

test: True — /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_voc/ImageSets/Main/test.txt
test_fog: True — /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_voc/ImageSets/Main/test_fog.txt
test_rain: True — /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_voc/ImageSets/Main/test_rain.txt
test_snow: True — /content/drive/MyDrive/Dissertation/Datasets/processed/acdc_voc/ImageSets/Main/test_snow.txt


# Register ACDC evaluation datasets

In [7]:
acdc_datasets = {
    "acdc_test": "test",
    "acdc_test_fog": "test_fog",
    "acdc_test_rain": "test_rain",
    "acdc_test_snow": "test_snow",
}

def reset_detectron2_dataset(name):
    if name in DatasetCatalog.list():
        DatasetCatalog.remove(name)

    if name in MetadataCatalog.list():
        MetadataCatalog.remove(name)

for dataset_name in acdc_datasets:
    reset_detectron2_dataset(dataset_name)

for dataset_name, split_name in acdc_datasets.items():
    register_pascal_voc(
        name=dataset_name,
        dirname=str(ACDC_VOC_ROOT),
        split=split_name,
        year="",
        class_names=CLASS_NAMES,
    )

print("ACDC datasets registered.")

for dataset_name in acdc_datasets:
    dataset_dicts = DatasetCatalog.get(dataset_name)
    metadata = MetadataCatalog.get(dataset_name)

    print(
        f"{dataset_name}: {len(dataset_dicts)} images | "
        f"classes: {metadata.thing_classes}"
    )

    assert metadata.thing_classes == CLASS_NAMES, (
        f"Class order mismatch for {dataset_name}"
    )

ACDC datasets registered.
acdc_test: 540 images | classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_fog: 100 images | classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_rain: 340 images | classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']
acdc_test_snow: 100 images | classes: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


# Load DAWN Model

The saved training configuration is reused so the architecture and parameters remain
consistent with the original DAWN in-domain experiment.

In [8]:
cfg = get_cfg()
cfg.merge_from_file(str(SOURCE_CONFIG))

if BEST_MODEL.exists():
    selected_weights = BEST_MODEL
    print("Using best model:", selected_weights)
else:
    selected_weights = FINAL_MODEL
    print("model_best.pth not found; using:", selected_weights)

cfg.MODEL.WEIGHTS = str(selected_weights)
cfg.MODEL.ROI_HEADS.NUM_CLASSES = len(CLASS_NAMES)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESHOLD

# The model was selected using DAWN validation during training.
# ACDC is used only now, as the independent target test dataset.
cfg.DATASETS.TEST = ("acdc_test",)
cfg.OUTPUT_DIR = str(OUTPUT_DIR)

predictor = DefaultPredictor(cfg)

print("Model loaded successfully.")
print("Score threshold:", cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST)
print("Number of classes:", cfg.MODEL.ROI_HEADS.NUM_CLASSES)

Using best model: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_indomain_fasterrcnn_r50fpn_seed42/model_best.pth
Model loaded successfully.
Score threshold: 0.5
Number of classes: 6


# Evaluate globally and by weather condition

In [9]:
eval_datasets = {
    "global": "acdc_test",
    "fog": "acdc_test_fog",
    "rain": "acdc_test_rain",
    "snow": "acdc_test_snow",
}

results_rows = []

for condition, dataset_name in eval_datasets.items():
    print(f"\nEvaluating DAWN → ACDC on {condition.upper()}...")

    condition_output = OUTPUT_DIR / "evaluation" / condition
    condition_output.mkdir(parents=True, exist_ok=True)

    evaluator = COCOEvaluator(
        dataset_name,
        cfg,
        False,
        output_dir=str(condition_output),
    )

    loader = build_detection_test_loader(cfg, dataset_name)
    eval_results = inference_on_dataset(
        predictor.model,
        loader,
        evaluator,
    )

    bbox = eval_results["bbox"]

    results_rows.append({
        "source_train_dataset": SOURCE_DATASET,
        "target_test_dataset": TARGET_DATASET,
        "model": "Faster R-CNN",
        "experiment": "DAWN->ACDC",
        "seed": RANDOM_SEED,
        "condition": condition,
        "mAP50-95": bbox["AP"] / 100,
        "mAP50": bbox["AP50"] / 100,
        "mAP75": bbox["AP75"] / 100,
    })

results_df = pd.DataFrame(results_rows)
results_df


Evaluating DAWN → ACDC on GLOBAL...


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0717 00:12:34.678000 13192 torch/fx/_symbolic_trace.py:53] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.


Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.151
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.264
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.151
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.040
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.168
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.288
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.126
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.205
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.205
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.052
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.211
 Average Recall     (AR) @[ IoU=0.50:0.

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.215
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.369
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.218
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.063
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.273
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.374
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.203
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.283
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.283
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.069
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.322
 Average Recall     (AR) @[ IoU=0.50:0.

Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.135
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.235
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.140
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.037
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.141
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.277
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.115
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.187
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.187
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.046
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.176
 Average Recall     (AR) @[ IoU=0.50:0.

,source_train_dataset,target_test_dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75
0,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,global,0.150596,0.264158,0.151190
1,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,fog,0.214656,0.369146,0.218382
2,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,rain,0.135165,0.234956,0.139549
3,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,snow,0.175870,0.307079,0.172668


# Measure inference time and FPS

In [10]:
def get_image_paths_from_split(voc_root, split_name):
    split_file = (
        voc_root
        / "ImageSets"
        / "Main"
        / f"{split_name}.txt"
    )

    with open(split_file, "r") as f:
        image_ids = [
            line.strip()
            for line in f
            if line.strip()
        ]

    image_paths = []

    for image_id in image_ids:
        jpg_path = voc_root / "JPEGImages" / f"{image_id}.jpg"
        png_path = voc_root / "JPEGImages" / f"{image_id}.png"

        if jpg_path.exists():
            image_paths.append(jpg_path)
        elif png_path.exists():
            image_paths.append(png_path)

    return image_paths


def measure_fps(predictor, image_paths, warmup=20):
    images = []

    for path in image_paths:
        image = cv2.imread(str(path))
        if image is not None:
            images.append(image)

    if not images:
        raise ValueError("No readable images were found for FPS measurement.")

    warmup_count = min(warmup, len(images))

    for image in images[:warmup_count]:
        _ = predictor(image)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    times = []

    for image in images:
        start = time.perf_counter()
        _ = predictor(image)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()
        times.append(end - start)

    inference_ms = float(np.mean(times) * 1000)
    fps = float(1000 / inference_ms)

    return inference_ms, fps


split_mapping = {
    "global": "test",
    "fog": "test_fog",
    "rain": "test_rain",
    "snow": "test_snow",
}

for idx, row in results_df.iterrows():
    condition = row["condition"]
    split_name = split_mapping[condition]

    image_paths = get_image_paths_from_split(
        ACDC_VOC_ROOT,
        split_name,
    )

    print(
        f"Measuring {condition}: "
        f"{len(image_paths)} images"
    )

    inference_ms, fps = measure_fps(
        predictor,
        image_paths,
        warmup=20,
    )

    results_df.loc[idx, "Inference_ms_per_image"] = inference_ms
    results_df.loc[idx, "FPS"] = fps

results_df

Measuring global: 540 images
Measuring fog: 100 images
Measuring rain: 340 images
Measuring snow: 100 images


,source_train_dataset,target_test_dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Inference_ms_per_image,FPS
0,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,global,0.150596,0.264158,0.151190,58.840446,16.995113
1,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,fog,0.214656,0.369146,0.218382,59.973425,16.674052
2,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,rain,0.135165,0.234956,0.139549,58.774611,17.014149
3,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,snow,0.175870,0.307079,0.172668,59.286444,16.867262


# Export results

In [11]:
results_csv = (
    OUTPUT_DIR
    / "dawn_to_acdc_faster_rcnn_seed42_results_summary.csv"
)

results_json = (
    OUTPUT_DIR
    / "dawn_to_acdc_faster_rcnn_seed42_results_summary.json"
)

results_df.to_csv(results_csv, index=False)

with open(results_json, "w") as f:
    json.dump(
        results_df.to_dict(orient="records"),
        f,
        indent=4,
    )

print("Saved CSV:", results_csv)
print("Saved JSON:", results_json)

print("\nDAWN → ACDC Faster R-CNN RESULTS")
display(results_df)

Saved CSV: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_to_acdc_faster_rcnn_seed42/dawn_to_acdc_faster_rcnn_seed42_results_summary.csv
Saved JSON: /content/drive/MyDrive/Dissertation/Runs/faster_rcnn/dawn_to_acdc_faster_rcnn_seed42/dawn_to_acdc_faster_rcnn_seed42_results_summary.json

DAWN → ACDC Faster R-CNN RESULTS


,source_train_dataset,target_test_dataset,model,experiment,seed,condition,mAP50-95,mAP50,mAP75,Inference_ms_per_image,FPS
0,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,global,0.150596,0.264158,0.151190,58.840446,16.995113
1,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,fog,0.214656,0.369146,0.218382,59.973425,16.674052
2,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,rain,0.135165,0.234956,0.139549,58.774611,17.014149
3,DAWN,ACDC,Faster R-CNN,DAWN->ACDC,42,snow,0.175870,0.307079,0.172668,59.286444,16.867262
